# 🚀 nanochat 双语大模型全流程训练 (Google Colab GPU)

包含：
- ⚙️ **Phase 1 推理优化**：双布局 KV Cache、增量流式解码、Backpressure
- ⚡ **Phase 2 训练优化**：ReLUSquare 激活算子融合、RoPE+QKNorm 编译、AdamW Cooldown $\beta_2$
- 🇨🇳 **Phase 3 中文全链路**：49,152 词表热扩充、SkyPile-150B 流式预训练、SmolTalkCN/GSM8KCN 对话微调、C-Eval 评测

> **注意**：请确保已在 **代码执行程序 -> 更改运行时类型** 中选择了 **GPU**（T4 / L4 / A100均可）。

### Step 1: 检查 GPU 环境与挂载 Google Drive

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

# 挂载 Google Drive (保存产出的模型权重与报告)
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/nanochat_run')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Google Drive 挂载就绪: {DRIVE_DIR}")

### Step 2: 克隆专属仓库 (chinese-support 分支) 并安装依赖

In [ ]:
%cd /content
!rm -rf nanochat
!git clone -b chinese-support https://github.com/Golenspade/nanochat.git
%cd /content/nanochat

# 使用 uv 安装依赖并激活 GPU 环境
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]
!uv venv /content/.venv
!uv sync --extra gpu
VENV_PY = "/content/.venv/bin/python"
!$VENV_PY -c "import torch; print('Virtualenv PyTorch CUDA ready:', torch.cuda.is_available())"

### Step 3: 准备数据集与训练中英双语扩展分词器 (Vocab 49152)

In [ ]:
!$VENV_PY -m nanochat.report reset

# 准备英文 Parquet 分片 (中文 SkyPile-150B 在训练时通过 HuggingFace 流式按比例加载)
!$VENV_PY -m nanochat.dataset -n 16

# 训练中文扩充 Tokenizer (vocab=49152, pattern=cn)
!$VENV_PY -m scripts.tok_train --vocab-size=49152 --pattern=cn
!$VENV_PY -m scripts.tok_eval

### Step 4: 启动中英文混合预训练 (Base Train)

In [ ]:
# 可按你的 GPU 规格调整参数：
# - depth=12 (~124M-150M 参数) 速度极快
# - device-batch-size: T4 设为 8~16, A100 设为 16~32
# - cn-ratio=0.5 (中英文各占 50% 混合数据流)

!$VENV_PY -m scripts.base_train \
    --depth=12 \
    --run=d12_cn \
    --model-tag=d12_cn \
    --cn-ratio=0.5 \
    --device-batch-size=16 \
    --window-pattern=L \
    --core-metric-every=999999 \
    --sample-every=-1 \
    --save-every=-1

### Step 5: 对话微调 (SFT Fine-Tuning)

In [ ]:
# 下载对话身份设定集
!curl -L -o ~/.cache/nanochat/identity_conversations.jsonl \
    https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl

# 执行多任务 SFT (SmolTalkCN 对话 + GSM8KCN 中文数学多步推理)
!$VENV_PY -m scripts.chat_sft \
    --run=d12_cn \
    --model-tag=d12_cn \
    --device-batch-size=16 \
    --cn-talk-epochs=1 \
    --gsm8k-cn-epochs=1 \
    --ceval-epochs=0

### Step 6: 双语综合评测 (English ChatCORE + Chinese C-Eval/GSM8K)

In [ ]:
# 英文基准评测
!$VENV_PY -m scripts.chat_eval -i sft -g d12_cn

# 中文综合评测 (C-Eval 多学科分类 + GSM8K-CN 推理)
!$VENV_PY -m scripts.eval_cn -i sft -g d12_cn -a 'CEval|GSM8K-CN' --ceval-split val

# 汇总生成可视化 HTML 报告
!$VENV_PY -m nanochat.report generate

### Step 7: 同步模型成果至 Google Drive 持久保存

In [ ]:
!cp -r ~/.cache/nanochat /content/drive/MyDrive/nanochat_run/
print("🎉 全部模型 Checkpoint、分词器与 HTML 评测报告已成功备份至 Google Drive！")
!ls -la /content/drive/MyDrive/nanochat_run/nanochat